# Source-aware query examples
Read-only examples for the current app. The original notebook is preserved in `docs/historical/prototype_ingestion.ipynb`. Initialize/import the catalog through the CLI before running these cells.

# Film Reel-ational Data — Prototyping & Schema Mapping Notebook
**Authors:** Andrew & Isaac  
**Milestones Covered:**
- Data ingestion (CSV / JSON column parsing)
- Mapping Local Schemas (Kaggle CSVs) to Global Relational Schema (SQLite)
- Conversion to SQLite database (`movies.db`)
- Verification queries for Multi-Table Search, CRUD, **Bombed at the Box Office**, and **Hidden Gems**

## 1. Local Schema to Global Schema Mapping
| Global SQLite Table | Global Attributes | Local Source (Kaggle CSVs) | Transformation / Constraints |
|---|---|---|---|
| `Movies` | `movie_id (PK)`, `title`, `budget`, `revenue`, `vote_average`, `vote_count` | `movies_metadata.csv` / `tmdb_5000_movies.csv` (`id`, `title`, `budget`, `revenue`, `vote_average`, `vote_count`) | `CHECK (vote_average BETWEEN 0 AND 10)` |
| `Actors` | `actor_id (PK)`, `actor_name` | `credits.csv` / `tmdb_5000_credits.csv` (`cast` JSON array -> `id`, `name`) | Deduplicated on `actor_id` |
| `Movie_Cast` | `movie_id (FK)`, `actor_id (FK)`, `character_name`, `PK(movie_id, actor_id)` | `credits.csv` (`id` + `cast` JSON array -> `id`, `character`) | `ON DELETE CASCADE` |
| `Genre_Types` | `genre_id (PK)`, `genre_name` | `movies_metadata.csv` (`genres` JSON array -> `id`, `name`) | 3NF Lookup Table |
| `Genres` | `movie_id (FK)`, `genre_id (FK)`, `PK(movie_id, genre_id)` | `movies_metadata.csv` (`id` + `genres` JSON array -> `id`) | `ON DELETE CASCADE` |

In [ ]:
from pathlib import Path
import sys
# This notebook can run from src/ or the repository root. Imports never rebuild data.
root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(root))
from src import database as db, validation
from src.ingest_data import DATA_DIR
print("Canonical input directory:", DATA_DIR)
with db.connection() as conn:
    print(db.stats(conn, "movies"))


In [ ]:
# Source-aware actor / role search on the existing catalog.
with db.connection() as conn:
    filters = validation.query({"actor": "Cillian Murphy"}, "index", conn)
    result = db.query_movies(conn, "index", filters)
for movie in result["rows"]:
    print(movie["display_title"], movie["vote_average"], movie["vote_count"])
print(result["sql"])


In [ ]:
# Known revenue below known production budget, using one selected source.
with db.connection() as conn:
    filters = validation.query({"min_budget": "50000000"}, "bombs", conn)
    result = db.query_movies(conn, "bombs", filters)
print("All qualifying movies:", result["total"])
print("Full-set summary:", result["summary"])
for movie in result["rows"][:5]:
    print(movie["display_title"], movie["net_profit"])


In [ ]:
# Dashboard and Hidden Gems share these defaults.
with db.connection() as conn:
    filters = validation.query({}, "gems", conn)
    result = db.query_movies(conn, "gems", filters)
    assert result["total"] == db.stats(conn, filters["source"])["gems"]
for movie in result["rows"][:5]:
    print(movie["display_title"], movie["gem_score"])
